# EGregoRA avatar neural-voice server (Colab prototype)

This notebook moves the avatars' existing **HeadTTS/Kokoro 82M timestamped** engine off visitors' devices. The model is cached in this Google Drive account and loaded by the Colab runtime; a visitor receives generated audio, word timings, phonemes, and visemes rather than the model weights.

It preserves the deployed male range: British `bm_george`, `bm_fable`, `bm_lewis`; American `am_fenrir`, `am_michael`, `am_puck`. HeadTTS 1.3.0 is pinned for compatibility with the sites and their lip-sync data.

> **Privacy change:** text submitted for speech is sent from the visitor's browser to this notebook. It is no longer wholly private, in-browser synthesis. This notebook does not intentionally log request bodies, but Colab, Cloudflare, and network providers process traffic. Do not submit secrets or sensitive personal data.

> **Prototype only:** Colab sessions sleep, disconnect, and expire; Quick Tunnel URLs rotate; accelerators are not guaranteed; and a public endpoint may be abused. Do not treat this notebook as a production service or use several accounts to evade Colab quotas. The existing sites should retain their native-browser fallback.


## How to run

1. Open this file in Google Colab and choose **Runtime → Run all**. A CPU runtime is sufficient.
2. Authorise the Google Drive mount. The first run downloads the model into `MyDrive/EGregoRA/headtts-model-cache`; later sessions reuse it.
3. By default, the tunnel cell creates a temporary `trycloudflare.com` URL. Copy the printed **HeadTTS REST base URL** (it ends in `/v1`).
4. Leave the final keep-alive cell running while the prototype is in use. Stopping or losing the runtime stops synthesis.

For an already configured Cloudflare **named tunnel**, add Colab secrets `CLOUDFLARE_TUNNEL_TOKEN` and `HEADTTS_PUBLIC_ORIGIN` (for example, `https://speech.example.org`). Never paste a tunnel token into a code cell or commit it to the notebook. A named hostname is stable, but the Colab service behind it is still ephemeral.


In [ ]:
# Runtime configuration — these are not credentials.
from pathlib import Path

HEADTTS_VERSION = "v1.3.0"
KOKORO_JS_VERSION = "1.2.1"
HEADTTS_PORT = 8882       # Private model process; never tunneled directly.
API_PORT = 8000           # Guarded public API.
DTYPE = "fp32"       # Best quality; q8 is a smaller/faster alternative.
THREADS = 1             # HeadTTS recommends one CPU worker for this model.
VOICES = [
    "bm_george", "bm_fable", "bm_lewis",
    "am_fenrir", "am_michael", "am_puck",
]
WORKDIR = Path("/content/HeadTTS")
DRIVE_CACHE = Path("/content/drive/MyDrive/EGregoRA/headtts-model-cache")
print("Voices to preload:", ", ".join(VOICES))


In [ ]:
# Mount the selected Google account's Drive for a persistent model cache.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
print("Persistent cache:", DRIVE_CACHE)


In [ ]:
# Install the pinned server and obtain the exact six voice embeddings.
# The model itself is downloaded server-side when HeadTTS starts.
import os, shutil, subprocess, sys

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", HEADTTS_VERSION,
    "https://github.com/met4citizen/HeadTTS.git", str(WORKDIR),
], check=True)
subprocess.run(["npm", "install"], cwd=WORKDIR, check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "fastapi>=0.115,<1", "uvicorn>=0.34,<1", "httpx>=0.28,<1",
], check=True)

# Redirect Transformers.js downloads to Drive before the server starts.
runtime_cache = WORKDIR / "node_modules/@huggingface/transformers/.cache"
if runtime_cache.is_symlink() or runtime_cache.is_file():
    runtime_cache.unlink()
elif runtime_cache.exists():
    shutil.rmtree(runtime_cache)
runtime_cache.parent.mkdir(parents=True, exist_ok=True)
runtime_cache.symlink_to(DRIVE_CACHE, target_is_directory=True)

# kokoro-js distributes the canonical embeddings used by the deployed selectors.
pack_name = subprocess.check_output(
    ["npm", "pack", f"kokoro-js@{KOKORO_JS_VERSION}", "--silent"],
    cwd=WORKDIR, text=True,
).strip().splitlines()[-1]
subprocess.run(["tar", "-xzf", pack_name], cwd=WORKDIR, check=True)
voice_dir = WORKDIR / "voices"
voice_dir.mkdir(exist_ok=True)
for voice in VOICES:
    source = WORKDIR / "package/voices" / f"{voice}.bin"
    target = voice_dir / source.name
    if not source.exists():
        raise FileNotFoundError(f"Missing canonical embedding: {source}")
    shutil.copy2(source, target)
shutil.rmtree(WORKDIR / "package")
(WORKDIR / pack_name).unlink()
print("Installed HeadTTS", HEADTTS_VERSION, "and", len(VOICES), "voices.")


In [ ]:
# Write a minimal server configuration. REST responses include WAV audio plus
# word timings, phonemes, and visemes used by the avatar lip sync.
import json

config = {
    "server": {
        "port": HEADTTS_PORT,
        "certFile": None,
        "keyFile": None,
        "websocket": False,
        "rest": True,
        "connectionTimeout": 20000,
        # This raw model port remains on loopback behind the guarded API.
        "corsOrigin": None,
    },
    "tts": {
        "threads": THREADS,
        "transformersModule": "@huggingface/transformers",
        "model": "onnx-community/Kokoro-82M-v1.0-ONNX-timestamped",
        "dtype": DTYPE,
        "device": "cpu",
        "styleDim": 256,
        "audioSampleRate": 24000,
        "frameRate": 40,
        "languages": ["en-us"],
        "dictionaryPath": "./dictionaries",
        "voicePath": "./voices",
        "voices": VOICES,
        "deltaStart": -10,
        "deltaEnd": 10,
        "defaults": {
            "voice": "bm_george",
            "language": "en-us",
            "speed": 1,
            "audioEncoding": "wav",
        },
    },
    "trace": 0,
}
CONFIG_PATH = WORKDIR / "egregora-headtts.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2) + "\n")
print(CONFIG_PATH.read_text())


In [ ]:
# Start HeadTTS, wait for the model, then warm every selected voice.
import requests, signal, subprocess, sys, textwrap, time

for process_name in ("headtts-node.mjs", "uvicorn guarded_api:app"):
    subprocess.run(["pkill", "-f", process_name], check=False)
SERVER_LOG = Path("/content/egregora-headtts.log")
server_log_handle = SERVER_LOG.open("w")
server_process = subprocess.Popen(
    ["node", "./modules/headtts-node.mjs", "--config", str(CONFIG_PATH)],
    cwd=WORKDIR, stdout=server_log_handle, stderr=subprocess.STDOUT,
    start_new_session=True,
)
local_headtts_base = f"http://127.0.0.1:{HEADTTS_PORT}/v1"
deadline = time.time() + 600
while time.time() < deadline:
    if server_process.poll() is not None:
        raise RuntimeError("HeadTTS exited early:\n" + SERVER_LOG.read_text()[-5000:])
    try:
        response = requests.post(f"{local_headtts_base}/hello", timeout=3)
        if response.ok and response.text.startswith("HeadTTS"):
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError("HeadTTS did not become ready within ten minutes.")

for voice in VOICES:
    started = time.time()
    response = requests.post(
        f"{local_headtts_base}/synthesize",
        json={
            "input": "Voice ready.", "voice": voice,
            "language": "en-us", "speed": 1, "audioEncoding": "wav",
        },
        timeout=300,
    )
    response.raise_for_status()
    body = response.json()
    if not body.get("audio") or not body.get("visemes"):
        raise RuntimeError(f"Warm-up failed for {voice}: malformed response")
    print(f"Warmed {voice} in {time.time() - started:.1f}s")
print("All six voices are loaded in the Colab runtime.")

# Place a guarded, origin-restricted API in front of HeadTTS. The model port
# remains private on loopback. The guard validates requests, caps text/body
# size, applies per-client rate limiting, and serialises expensive inference.
GUARD_SOURCE = r'''import asyncio
import time
from collections import defaultdict, deque
from typing import Any, Literal

import httpx
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, PlainTextResponse, Response
from pydantic import BaseModel, ConfigDict, Field

ALLOWED_VOICES = {
    "bm_george", "bm_fable", "bm_lewis",
    "am_fenrir", "am_michael", "am_puck",
}
ALLOWED_ORIGINS = [
    "https://the-two-infinities.hatchable.site",
    "https://egregora.hatchable.site",
]
HEADTTS_URL = "http://127.0.0.1:8882/v1/synthesize"
MAX_BODY_BYTES = 32_000
MAX_TEXT_CHARS = 500
MAX_ITEMS = 64
RATE_WINDOW_SECONDS = 60
RATE_REQUESTS = 12
MAX_RESPONSE_BYTES = 16_000_000

class SynthesisBody(BaseModel):
    model_config = ConfigDict(extra="forbid")
    input: str | list[Any]
    voice: Literal[
        "bm_george", "bm_fable", "bm_lewis",
        "am_fenrir", "am_michael", "am_puck",
    ]
    language: Literal["en-us"] = "en-us"
    speed: float = Field(default=1.0, ge=0.6, le=1.4)
    audioEncoding: Literal["wav"] = "wav"

app = FastAPI(title="EGregoRA guarded HeadTTS prototype", docs_url=None, redoc_url=None)
app.add_middleware(
    CORSMiddleware,
    allow_origins=ALLOWED_ORIGINS,
    allow_methods=["POST", "GET", "OPTIONS"],
    allow_headers=["Content-Type"],
    max_age=600,
)
hits: dict[str, deque[float]] = defaultdict(deque)
tts_lock = asyncio.Semaphore(1)

def validate_input(value: str | list[Any]) -> None:
    if isinstance(value, str):
        chars = len(value.strip())
        if chars < 1 or chars > MAX_TEXT_CHARS:
            raise HTTPException(422, f"input must contain 1-{MAX_TEXT_CHARS} characters")
        return
    if not isinstance(value, list) or not value or len(value) > MAX_ITEMS:
        raise HTTPException(422, f"input array must contain 1-{MAX_ITEMS} items")
    chars = 0
    total_break_ms = 0.0
    for item in value:
        if isinstance(item, str):
            chars += len(item)
            continue
        if not isinstance(item, dict) or set(item) != {"type", "value"}:
            raise HTTPException(422, "each input item must contain only type and value")
        kind = item["type"]
        item_value = item["value"]
        if kind == "text" and isinstance(item_value, str):
            chars += len(item_value)
        elif kind == "break" and isinstance(item_value, (int, float)) and 0 <= item_value <= 5000:
            total_break_ms += float(item_value)
        elif kind in {"date", "time"} and isinstance(item_value, (int, float)) and item_value >= 0:
            chars += 10
        else:
            raise HTTPException(422, "unsupported or invalid input item")
    if chars < 1 or chars > MAX_TEXT_CHARS:
        raise HTTPException(422, f"input must contain 1-{MAX_TEXT_CHARS} text characters")
    if total_break_ms > 15_000:
        raise HTTPException(422, "combined breaks may not exceed 15 seconds")

def rate_limit(request: Request) -> None:
    key = request.headers.get("cf-connecting-ip") or (request.client.host if request.client else "unknown")
    now = time.monotonic()
    bucket = hits[key]
    while bucket and bucket[0] <= now - RATE_WINDOW_SECONDS:
        bucket.popleft()
    if len(bucket) >= RATE_REQUESTS:
        raise HTTPException(429, "prototype request limit reached; retry shortly")
    bucket.append(now)

@app.middleware("http")
async def cap_body(request: Request, call_next):
    length = request.headers.get("content-length")
    if length and int(length) > MAX_BODY_BYTES:
        return JSONResponse({"error": "request body too large"}, status_code=413)
    return await call_next(request)

@app.get("/healthz")
async def health():
    return {"status": "ok", "engine": "HeadTTS 1.3.0", "voices": sorted(ALLOWED_VOICES)}

@app.post("/v1/hello", response_class=PlainTextResponse)
async def hello():
    return "HeadTTS v1.3.0 — EGregoRA guarded Colab prototype"

@app.get("/v1/voices")
async def voices():
    return {"voices": sorted(ALLOWED_VOICES), "sampleRate": 24000, "format": "wav"}

@app.post("/v1/synthesize")
async def synthesize(payload: SynthesisBody, request: Request):
    rate_limit(request)
    validate_input(payload.input)
    try:
        async with tts_lock:
            async with httpx.AsyncClient(timeout=300) as client:
                upstream = await client.post(HEADTTS_URL, json=payload.model_dump())
    except httpx.HTTPError as exc:
        raise HTTPException(502, "neural synthesis process is unavailable") from exc
    if upstream.status_code != 200:
        raise HTTPException(502, "neural synthesis process rejected the request")
    if len(upstream.content) > MAX_RESPONSE_BYTES:
        raise HTTPException(502, "neural synthesis response exceeded the safety limit")
    return Response(content=upstream.content, media_type="application/json")
'''
GUARD_PATH = Path("/content/guarded_api.py")
GUARD_PATH.write_text(textwrap.dedent(GUARD_SOURCE))
API_LOG = Path("/content/egregora-guarded-api.log")
api_log_handle = API_LOG.open("w")
api_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn", "guarded_api:app",
        "--app-dir", "/content", "--host", "127.0.0.1",
        "--port", str(API_PORT), "--no-access-log",
    ],
    stdout=api_log_handle, stderr=subprocess.STDOUT, start_new_session=True,
)
local_api_base = f"http://127.0.0.1:{API_PORT}"
deadline = time.time() + 60
while time.time() < deadline:
    if api_process.poll() is not None:
        raise RuntimeError("Guarded API exited early:\n" + API_LOG.read_text()[-5000:])
    try:
        response = requests.post(f"{local_api_base}/v1/hello", timeout=3)
        if response.ok and response.text.startswith("HeadTTS"):
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise TimeoutError("Guarded API did not become ready within one minute.")
print("Guarded API ready; the raw model port remains private.")


## Publish the temporary endpoint

The next cell downloads the official `cloudflared` binary and opens either the configured named tunnel or a temporary Quick Tunnel. No Cloudflare account is required for a Quick Tunnel. Its URL changes whenever the process restarts.

Anyone who knows the URL can call it. A guarded proxy allows only the two deployed site origins in browser CORS, the six selected voices, WAV output, conservative speed, at most 500 text characters per request, and 12 requests per client IP per minute. It also limits body/response sizes and serialises synthesis. These controls reduce accidental abuse but are not authentication or production-grade denial-of-service protection. Do not publish a Quick Tunnel URL broadly, and stop the runtime when testing is complete.


In [ ]:
# Open a Cloudflare tunnel without writing credentials into this notebook.
import re, stat, subprocess, time
from urllib.request import urlretrieve

CLOUDFLARED = Path("/content/cloudflared")
if not CLOUDFLARED.exists():
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IXUSR)
subprocess.run(["pkill", "-f", str(CLOUDFLARED)], check=False)

try:
    from google.colab import userdata
    try:
        tunnel_token = userdata.get("CLOUDFLARE_TUNNEL_TOKEN")
    except Exception:
        tunnel_token = None
    try:
        named_origin = userdata.get("HEADTTS_PUBLIC_ORIGIN")
    except Exception:
        named_origin = None
except ImportError:
    tunnel_token = named_origin = None

TUNNEL_LOG = Path("/content/egregora-cloudflared.log")
tunnel_log_handle = TUNNEL_LOG.open("w")
if tunnel_token:
    if not named_origin:
        raise RuntimeError(
            "A named tunnel token was found, but HEADTTS_PUBLIC_ORIGIN is missing. "
            "Add that Colab secret with the configured https:// hostname."
        )
    tunnel_command = [str(CLOUDFLARED), "tunnel", "--no-autoupdate", "run", "--token", tunnel_token]
    public_origin = named_origin.rstrip("/")
else:
    tunnel_command = [
        str(CLOUDFLARED), "tunnel", "--no-autoupdate",
        "--url", f"http://127.0.0.1:{API_PORT}",
    ]
    public_origin = None

tunnel_process = subprocess.Popen(
    tunnel_command, stdout=tunnel_log_handle, stderr=subprocess.STDOUT,
    start_new_session=True,
)
if public_origin is None:
    deadline = time.time() + 90
    while time.time() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError("cloudflared exited early:\n" + TUNNEL_LOG.read_text()[-5000:])
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", TUNNEL_LOG.read_text())
        if match:
            public_origin = match.group(0)
            break
        time.sleep(1)
    else:
        raise TimeoutError("No Quick Tunnel URL appeared within 90 seconds.")

PUBLIC_REST_BASE = public_origin + "/v1"
print("\nHeadTTS REST base URL:", PUBLIC_REST_BASE)
print("Health check:", PUBLIC_REST_BASE + "/hello")
print("This URL works only while this Colab runtime and both processes remain active.")


In [ ]:
# Verify the public route and audition one server-generated result.
import base64, io, requests
from IPython.display import Audio, display

health = requests.post(PUBLIC_REST_BASE + "/hello", timeout=30)
health.raise_for_status()
print(health.text)
test = requests.post(
    PUBLIC_REST_BASE + "/synthesize",
    json={
        "input": "EGregoRA neural speech is running on the notebook, not in your browser.",
        "voice": "bm_george", "language": "en-us",
        "speed": 0.91, "audioEncoding": "wav",
    },
    timeout=300,
)
test.raise_for_status()
test_body = test.json()
print("Words:", len(test_body.get("words", [])), "Visemes:", len(test_body.get("visemes", [])))
display(Audio(base64.b64decode(test_body["audio"]), autoplay=False))

print("\nSite integration value (REST endpoint, including /v1):")
print(PUBLIC_REST_BASE)


## Connecting EGregoRA (separate integration step)

HeadTTS browser clients must use the printed value as their REST endpoint, including `/v1`. For example: `https://random-name.trycloudflare.com/v1`. The browser client then calls `/v1/hello` and `/v1/synthesize`.

Do **not** hard-code a temporary Quick Tunnel address into a production deployment: it becomes invalid when this runtime ends. A production switch should happen separately, after choosing a stable endpoint and accepting the privacy, uptime, quota, and abuse trade-offs. Keep the sites' native voice fallback and show users that neural text leaves their device.

This notebook deliberately does not contain an API key, admin passcode, Google credential, tunnel credential, or visitor voiceprint. It serves synthesis only; the podcast voiceprint system remains local and unrelated.


In [ ]:
# Keep the runtime visibly active. Interrupt this cell to stop monitoring;
# use Runtime → Disconnect and delete runtime when finished.
import datetime, time
while True:
    if server_process.poll() is not None:
        raise RuntimeError("HeadTTS stopped; inspect " + str(SERVER_LOG))
    if api_process.poll() is not None:
        raise RuntimeError("Guarded API stopped; inspect " + str(API_LOG))
    if tunnel_process.poll() is not None:
        raise RuntimeError("cloudflared stopped; inspect " + str(TUNNEL_LOG))
    print(datetime.datetime.now().isoformat(timespec="seconds"), "— service active at", PUBLIC_REST_BASE)
    time.sleep(300)
